# Can fine-tunes of known kind be put side by side at equal size on Qwen?

*E45, Qwen line, 2026-10-28 to 2026-10-30. Reports: `reports/analyses/2026-10-30_analysis_e45-amendment2.md`. Plan
`reports/plans/2026-10-28_plan_qwen-q2.md` with Amendments 1 and 2. Code and figures: `experiments/qwen_updates/`.*

## The answer

Not yet. Both anchors learned what they were taught: the invented facts by step 100, the answer template by step 19.
But the template anchor moves the model's maths answers only about half as far as the invented-facts anchor has
by step 100, the first saved state at which it recalls its facts; most of that anchor's movement there is a shorter
answer format. No saved state puts the two at the same size, so no decomposition reading can be validated against
them as planned. The fine-tunes on the model's own answers never lowered their held-out loss. Every check of the
apparatus passed, and the anchor design goes back to the user.

## What we were trying to test

The project asks whether a post-training update strengthens something a model can already do or teaches it something
new, and whether a parameter decomposition can tell the two apart. On Qwen3-1.7B that needs updates whose kind is
known by design, so each decomposition reading can first be checked against a right answer. E45 makes those updates.
Nothing is read through the decomposition here.

Every fine-tune changes all 196 decomposed weight matrices with one recipe. It runs 100 steps of 64 sequences, with
ten warm-up steps to a peak rate of 1e-5 and cosine decay after.

- **Anchor 1, "already there".** The model's own answers under a prompt, trained without the prompt. No prompt we
  tried made the model better at maths, so the prompt became a fixed answer template: a "Given:" line, a "Steps:"
  list, then "The answer is". Anchor 1 is a format fine-tune: it fixes the novelty of a surface format the model
  already follows on request, not of a better way to solve problems. The format prompt lowers the original model's
  GSM8K accuracy on the 512 search problems from 0.834 to 0.680, and shortens answers from 190 to 130 tokens.
  Anchor 1 shares its domain and its data generation with the GSM8K arms; anchor 2 does not.
- **Anchor 2, "new".** Question-answer pairs about 200 invented people, trained for 300 steps, two seeds that each
  draw their own people.
- **Gold.** GSM8K's human-written solutions.
- **Own answers.** The model's own correct GSM8K answers, at the filed rate and at twice it.
- **The null.** The model's own answers without the correctness filter. It shows what training on itself does alone.

Only the anchors carry predictions about the decomposition; the GSM8K arms are to be read against them.

**Size.** KL to start is the mean per-token KL divergence from the original model to the fine-tuned one,
$\mathrm{KL}(W_0 \,\|\, W_0 + \delta W)$; 0 means nothing changed. It is read on 128 shared maths rows: the original
model's own answers to GSM8K test problems. Amendment 2 reads every run where it has learned and its maths-row KL is
nearest the invented-facts anchor's at step 100, a target of 0.618. A run is matched within a factor of 1.5 either
way. Gold's step 100 is the one independent check that matching works.

**Learned.** For anchor 1, the share of the gap to the prompted model closed on held-out answers must reach 0.25;
0 is the original model and 1 the prompted one. For anchor 2, a quarter of held-out invented facts must be recalled.
For the own-answer runs and the null, held-out loss on correct answers must fall by 0.02 nats per token.

**First-minute checks.** A control repeats E40's one-step gold fine-tune and must reproduce its KL. At step 2, the
first step with a nonzero rate, every run's maths-row KL must stay small, which catches a missing warm-up.

## What we expected, and why

- Anchor 2 would learn its facts by step 100, helped by packing about eight facts into each training row.
- A prompt would raise the model's maths accuracy for anchor 1 to teach. Failing that, the template prompt would be
  followed on nine answers in ten with the prompt and almost never without.
- Each run would be compared where its KL on its own training text first reached 0.1. Amendment 2 replaced this
  with matching on the maths rows, the one axis all runs share.
- The own-answer run would raise GSM8K accuracy a little and keep its answer length; gold would halve its length.
- Amendment 2 left open whether anchor 1 reaches the matching band by step 100, and said a card goes to the user if
  it does not.

## What we observed: the first jobs and Amendment 1

### Has the invented-facts fine-tune learned at the equal-movement points?

![Anchor 2 over training](../experiments/qwen_updates/figures/log_anchor2.png)

**Reading it.** No. It crosses every equal-movement point by step 12 but recalls its facts only at step 100, on both
seeds. At step 12 it recalls about 2% of held-out facts; at step 100 about 91%. Its unprompted maths accuracy then
collapses, and a 4-shot prompt brings it mostly back. So its movement on the maths rows is mostly a shorter answer
format, not lost maths. This is why Amendment 2 reads anchor 2 at step 100 and matches every other run to it.

### Is there a prompt that makes the model better at maths, for anchor 1 to teach?

![Paired accuracy change per candidate prompt](../experiments/qwen_updates/figures/log_anchor1.png)

**Reading it.** No. None of the three prompts raised accuracy by the 0.03 the plan required, so anchor 1 fell back
to the answer template. The template prompt lowers accuracy. The amended rule, which forgives stray whitespace,
counts it followed on 93% of answers with the prompt and none without.

### Why did two fine-tunes stop at step 2?

![Step-2 KL against the old limit](../experiments/qwen_updates/figures/log_step2.png)

**Reading it.** Because the limit was set from the gold run alone. The own-answer run and the null train on the
model's own maths answers, which are the rows the check reads, so they sit just above it. Amendment 2 replaced the
limit with a tenth of one full-rate step taken with no warm-up, the failure the check exists to catch.

## Amendment 2: anchor 1 as the template, every run matched to anchor 2

*2026-10-30. Predictions `experiments/qwen_updates/results/predictions_amend2.json`, run brief `RUN_BRIEF_amend2.md`;
eleven one-L40 jobs on the MATS cluster, $0. Figures `experiments/qwen_updates/plot_amend2.py`.*

### Did every fine-tune start as the recipe intends?

![Step-2 KL for every run under the amended limit](../experiments/qwen_updates/figures/amend2_step2.png)

**Reading it.** Yes. Every dot sits under the solid limit line. The two template-anchor seeds land inside the
shaded range between gold and the null, as expected. The rerun own-answer and null fine-tunes reproduced their
stopped step-2 values to every printed digit. So nothing in the code or data changed between the stop and the rerun.

### Did the template anchor learn its template?

![Template following and gap closed over training](../experiments/qwen_updates/figures/amend2_anchor1.png)

**Reading it.** Yes, on both seeds. Without any prompt, about 95% of its answers follow the template at step 19,
level with the original model under the prompt. Before step 19 the share is exactly zero. At step 8 about half the
answers already open with "Given that" in prose, but none has the "Steps:" list. We recounted the template with an
independent jq version of the rule on the saved answers, and it gives the stored counts exactly. The gap to the
prompted model closes past the filed half by step 3. Its maths accuracy fell from 0.78 to about 0.65, toward the
prompted model's.

*Anchor 1 is a format fine-tune: it fixes the novelty of a surface format the model already follows on request, not
of a better way to solve problems. The format prompt lowers the original model's GSM8K accuracy on the 512 search
problems from 0.834 to 0.680, and shortens answers from 190 to 130 tokens. Anchor 1 shares its domain and its data
generation with the GSM8K arms; anchor 2 does not.*

### Does the template anchor reach the matching band by step 100?

![Maths-row KL along training against the matching band](../experiments/qwen_updates/figures/amend2_matching.png)

**Reading it.** No. The orange lines rise with the warm-up and level off near 0.33, below the green band, on both
seeds. Gold, the positive control, ends inside the band, and so do the two invented-facts seeds, by construction.
The template anchor stops moving once it has closed most of the gap to the prompted model. More steps of the same
recipe would likely not help, though this was not tested. Anchor 2 was not saved between steps 12 and 100, so a lower size where both have
learned is not ruled out.

*Anchor 1 is a format fine-tune: it fixes the novelty of a surface format the model already follows on request, not
of a better way to solve problems. The format prompt lowers the original model's GSM8K accuracy on the 512 search
problems from 0.834 to 0.680, and shortens answers from 190 to 130 tokens. Anchor 1 shares its domain and its data
generation with the GSM8K arms; anchor 2 does not.*

*Anchor 2's movement on the maths rows is mostly a format shift to one-line answers, not its content: with a 4-shot
prompt its maths accuracy at step 100 is within 0.016 of the original model's.*

### Did the own-answer fine-tunes and the null learn?

![Held-out loss change on own correct answers](../experiments/qwen_updates/figures/amend2_own_null.png)

**Reading it.** No, at no saved state of any of the three. Their held-out loss rises sharply in the warm-up and is
still above the start at step 100. The rule asks for a fall of 0.02 nats per token from a start of 0.057. On the
maths rows the same runs rise to about 0.3 and fall back to about 0.1, as the matching figure shows. Yet the
filed-rate run raised paired GSM8K accuracy by five points over the full test. The update changed behaviour that the
loss rule does not see.

## What we learned

Both anchors learn what they are taught, and the apparatus is sound: the control, the warm-up check and the
reproduction all held. But the template anchor's whole behaviour moves the model's maths answers only half as far as
the invented-facts anchor has moved them by step 100, the first saved state at which it recalls its facts. No saved state puts the two at one size, so the plan's way of
validating a decomposition reading cannot run. The own-answer fine-tunes never learn by the filed loss rule, so they
cannot be read at the matched point either. For `GOALS.md`, E45 has no validated anchor pair for any reading yet.

*Anchor 1 is a format fine-tune: it fixes the novelty of a surface format the model already follows on request, not
of a better way to solve problems. The format prompt lowers the original model's GSM8K accuracy on the 512 search
problems from 0.834 to 0.680, and shortens answers from 190 to 130 tokens. Anchor 1 shares its domain and its data
generation with the GSM8K arms; anchor 2 does not.*

*Anchor 2's movement on the maths rows is mostly a format shift to one-line answers, not its content: with a 4-shot
prompt its maths accuracy at step 100 is within 0.016 of the original model's.*

## How compelling is the evidence

- The template anchor's shortfall holds on two seeds with different samples. Its distance below the band is about
  twenty times the seed difference.
- The tracker read at every step to 30 and every five after, so no crossing between saved states was missed.
- KL is read on one set of rows, the original model's own maths answers. Each run's own text gives a different axis
  that does not compare across runs.
- Gold is the only independent check of the matching. The invented-facts seeds match by construction.
- The own-answer runs and the null have one seed each.

## Surprises

- **Training on its own correct answers made held-out loss on such answers worse at every saved step.** The plan
  expected its canonical update to learn. The loss rose sharply in the warm-up and stayed above the start, at both
  rates and for the null, and the training loss rose too. Yet accuracy rose five points at the filed rate. We have
  not worked out why.
- The template anchor's KL grows from 0.258 at step 10 to about 0.32 by step 27, then creeps to 0.33, while its gap
  closed keeps rising.
- The template appears all at once between steps 8 and 19.

*Anchor 1 is a format fine-tune: it fixes the novelty of a surface format the model already follows on request, not
of a better way to solve problems. The format prompt lowers the original model's GSM8K accuracy on the 512 search
problems from 0.834 to 0.680, and shortens answers from 190 to 130 tokens. Anchor 1 shares its domain and its data
generation with the GSM8K arms; anchor 2 does not.*

## What next

1. **A card to the user on the anchor design**, as the amendment says. There are three choices. Read each anchor
   where it has learned, unmatched, with its size stated; this gives readings but no validation under the filed rule.
   Save and read anchor 2 between steps 12 and 100, to look for a lower shared size. Or replace anchor 1 with a
   behaviour that moves the maths rows further.
2. **The own-answer arms' learned rule.** A behaviour rule such as paired accuracy would count the filed-rate run
   as learned. Changing it changes a filed rule after the data, so it is the user's call.
3. E57 reads the saved states for $0; its anchor-1 and own-answer readings carry this report's labels.

## Departures from the literature

- **Context distillation.** The literature distils a prompt by matching the prompted model's full output
  distribution. We train on sampled answers instead, to keep one recipe for every arm.
- **The distilled behaviour** is a format that lowers accuracy, not an improving prompt, because no improving prompt
  qualified.
- **The template rule** was loosened after the answers were seen. It is a filed rule change.
- **Rejection-sampling fine-tuning** trains several epochs on many de-duplicated paths. We train 100 steps on four
  samples per problem.
- **The recipe** follows LLaMA-Factory's full-SFT setting for Qwen but stops at 100 steps, before one epoch.
- **Matching on KL** has no literature source. Its 1.5 factor was chosen with gold's value already known.

## Scorecard

| id | claim in words | read | verdict |
|---|---|---|---|
| own_accuracy | own answers raise paired accuracy ≥ 0.01 | +0.052 on 1,220 problems | holds |
| own_length | own answers keep ≥ 0.9 of length | 1.029 | holds |
| gold_length | gold ≤ 0.8 of length | 0.478 | holds |
| anchor2_learns | 100-step run recalls ≥ 0.5 | seed 0: 0.319 | fails; replaced by Amendment 1 |
| amend1 anchor2_learns | 300-step run recalls ≥ 0.5 | 0.999 and 1.0 | holds, both seeds |
| format_follow | template rule reproduces 474 and 0 of 512; qualifies | 0.926 and 0 | holds |
| anchor1_learns | gap closed ≥ 0.5 at step 100 | 0.942 and 0.943 | holds, both seeds |
| anchor1_limits | three limit sentences beside every anchor-1 reading | beside each anchor-1 section here | done |
| decisive_point gold | positive control matched | step 100, 0.697 | holds |
| decisive_point anchor 2 | both seeds at step 100 | 0.629 and 0.607 | holds |
| decisive_point anchor 1 | matched within 0.412-0.927 | 0.333 and 0.337 | not matched: no row validated, card |
| decisive_point own, own2, null | read at a learned state | none learned | not read |
| early checks | band, step-2 limit, reproduction, kept ≥ 4,000 | all within; 5,939 kept | hold |

Full scoring, rule by rule: `reports/analyses/2026-10-30_analysis_e45-amendment2.md`.

## Not run

- Any decomposition reading. E57 reads the saved states.
- Anchor 2's fact recall between steps 12 and 100.
- Anchor 1 past step 100 or at a higher rate.
- KL on the maths rows between the original model with and without the template prompt, the most the copied
  behaviour could move those rows.
- Why the own-answer runs' loss rises; a second seed of those runs.

**Provenance.** MATS cluster, `debug` QoS, one L40 per job, $0 throughout. Results in
`experiments/qwen_updates/results/`: `control.json`, `P.json`, `S_*.json`, `T_*.json`, `E_*.json`, `score.json`,
`run.log`. δW states in `results/large/`, a link into `~/commute-data/results-large/qwen_updates`.